In [0]:
-- -----------------------------------------------------------------------------
-- 1. BOOTSTRAP — create fact_sales shell (run once manually)
-- -----------------------------------------------------------------------------
 
-- CREATE TABLE IF NOT EXISTS workspace.fact.sales (
--       DATE_ID         INT            NOT NULL   -- format YYYYMMDD e.g. 20240115
--     , TRANSACTION_ID  INT            NOT NULL   -- unique row number, PK
--     , PRODUCT_ID      INT            NOT NULL   -- FK → dim_product
--     , SITE_ID         INT            NOT NULL   -- FK → dim_site
--     , QUANTITY        INT            NOT NULL   -- units sold in this transaction
--     , UNIT_PRICE      DECIMAL(10, 2) NOT NULL   -- price per unit
--     , SALES           DECIMAL(12, 2) NOT NULL   -- quantity × unit_price
-- )
--  USING DELTA
--  PARTITIONED BY (DATE_ID)
--  COMMENT 'Daily sales fact table — incremental append, one day added per run';

-- DROP TABLE workspace.fact.sales;

-- -----------------------------------------------------------------------------
-- 1. TRUNCATE — clear all existing data ahead of the reload
-- -----------------------------------------------------------------------------

--TRUNCATE TABLE workspace.fact.sales;


-- -----------------------------------------------------------------------------
-- 1. FULL REFRESH INSERT — reload last 3 years up to yesterday
-- -----------------------------------------------------------------------------
INSERT INTO workspace.fact.sales

WITH cteDates AS
(
    SELECT DISTINCT DATE_ID
    FROM workspace.reference.dates
    WHERE 1=1
--        AND DATE_ID BETWEEN '2024-01-01' AND date_sub(current_date(), 1)
        AND DATE_ID = date_sub(current_date(), 1)
)
, cteDateProduct AS
(
    SELECT
          D.DATE_ID
        , P.PRODUCT_ID
    FROM cteDates D
    CROSS JOIN workspace.dimension.dim_product P
)
, cteDateProductSite AS
(
    SELECT
          DP.DATE_ID
        , DP.PRODUCT_ID
        , S.SITE_ID
    FROM cteDateProduct DP
    CROSS JOIN workspace.dimension.dim_site S
)
-- Explode each date + product + site into a random number of transactions (1–10).
-- The number of transactions is deterministic per combination via HASH,
-- so re-running always produces the same set of rows.
, cteTransactions AS
(
    SELECT
          DATE_ID
        , PRODUCT_ID
        , SITE_ID
        , TRANSACTION_ID
    FROM cteDateProductSite DPS
    CROSS JOIN LATERAL (
        SELECT explode(sequence(1,
            (ABS(HASH(concat(DPS.DATE_ID, '-', DPS.PRODUCT_ID, '-', DPS.SITE_ID, '-count'))) % 10) + 1
        )) AS TRANSACTION_ID
    ) T
)
, cteBackfill AS
(
    SELECT
        -- Globally unique sequential integer ordered by date, product, site, txn
          ROW_NUMBER() OVER (ORDER BY DATE_ID, PRODUCT_ID, SITE_ID, TRANSACTION_ID) AS TRANSACTION_ID
        , CAST(date_format(DATE_ID, 'yyyyMMdd') AS INT)                              AS DATE_ID
        , PRODUCT_ID
        , SITE_ID
        -- Quantity: 1–20 units, varies per transaction
        , CAST(
            (ABS(HASH(concat(DATE_ID, '-', PRODUCT_ID, '-', SITE_ID, '-', TRANSACTION_ID, '-qty'))) % 20) + 1
          AS INT)                                                                     AS QUANTITY
        -- Unit price: product base price ± small variance per transaction
        , ROUND(
            CASE PRODUCT_ID
                WHEN 1 THEN 299.99   -- Gadget X
                WHEN 2 THEN 249.99   -- Gadget Y
                WHEN 3 THEN 199.99   -- Gadget Z
                WHEN 4 THEN  49.99   -- Widget A
                WHEN 5 THEN  39.99   -- Widget B
                WHEN 6 THEN  29.99   -- Widget C
                ELSE         99.99
            END
            -- ±10% price variance per transaction
            * (0.90 + (ABS(HASH(concat(DATE_ID, '-', PRODUCT_ID, '-', SITE_ID, '-', TRANSACTION_ID, '-price'))) % 21) * 0.01)
          , 2)                                                                        AS UNIT_PRICE
    FROM cteTransactions
)
-- Derive sales as quantity × unit_price
, cteFinal AS
(
    SELECT
          DATE_ID
        , TRANSACTION_ID
        , PRODUCT_ID
        , SITE_ID
        , QUANTITY
        , UNIT_PRICE
        , ROUND(QUANTITY * UNIT_PRICE, 2) AS SALES
    FROM cteBackfill
)
SELECT *
FROM cteFinal
ORDER BY 1 DESC;